## sklearn tfidf

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np

# 1. Load the data
df = pd.read_csv("./enron_spam_data.csv")

# do some sampling, good to see what happens as you sample more and more
df = df.sample(frac=0.5)

df['text'] = df['Subject'].fillna(' ') + df['Message'].fillna(' ')

# dirty trick to make the labels int 0/1
df['labels'] = (df['Spam/Ham']=='spam')*1

df = df[['text', 'labels']]


# 2. Split the dataset

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['labels'], test_size=0.2, random_state=37)

print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

# --- 3. Feature Extraction using TfidfVectorizer ---
# Initialize the TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2)

# Fit the vectorizer on the training data and transform both training and testing data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\nShape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")
print(f"Number of unique tokens: {len(tfidf_vectorizer.vocabulary_)}")

# --- 4. Train a Logistic Regression Classifier ---
# Initialize the Logistic Regression model
model = LogisticRegression(solver='liblinear', random_state=37)

# Train the model on the TF-IDF transformed training data
model.fit(X_train_tfidf, y_train)

# --- 5. Make Predictions on the Test Set ---
y_pred = model.predict(X_test_tfidf)

# --- 6. Evaluate the Model ---
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy on the test set: {accuracy:.4f}")

f1  = f1_score(y_test, y_pred)
print(f"\f1 on the test set: {f1:.4f}")


print("\nClassification Report:")
print(classification_report(y_test, y_pred))



# --- 7. Example Prediction on New Emails ---
new_emails = [
    "Subject: Free money! Claim your prize now!",
    "Hi John, please find attached the report you requested.",
    "Urgent: Your account has been compromised. Click here to secure it.",
    "Meeting reminder for tomorrow at 10 AM.",
    "Buy one get one free on all our products!"
]

new_emails_tfidf = tfidf_vectorizer.transform(new_emails)
new_predictions = model.predict(new_emails_tfidf)

print("\nPredictions on new emails:")
for email, prediction in zip(new_emails, new_predictions):
    print(f"Email: '{email}' -> Prediction: {'Spam' if prediction == 1 else 'Ham'}")

#  Feature strengths - look at top ones
feature_names = tfidf_vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

# Get the top N most important features for spam and ham
top_n = 15
top_spam_features_indices = coefficients.argsort()[-top_n:][::-1]
top_ham_features_indices = coefficients.argsort()[:top_n]

print(f"\nTop {top_n} features associated with Spam:")
for index in top_spam_features_indices:
    print(f"- {feature_names[index]}: {coefficients[index]:.4f}")

print(f"\nTop {top_n} features associated with Ham:")
for index in top_ham_features_indices:
    print(f"- {feature_names[index]}: {coefficients[index]:.4f}")




Training set size: 13486
Testing set size: 3372

Shape of X_train_tfidf: (13486, 51352)
Shape of X_test_tfidf: (3372, 51352)
Number of unique tokens: 51352

Accuracy on the test set: 0.9834
1 on the test set: 0.9839

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98      1654
           1       0.97      1.00      0.98      1718

    accuracy                           0.98      3372
   macro avg       0.98      0.98      0.98      3372
weighted avg       0.98      0.98      0.98      3372


Predictions on new emails:
Email: 'Subject: Free money! Claim your prize now!' -> Prediction: Spam
Email: 'Hi John, please find attached the report you requested.' -> Prediction: Ham
Email: 'Urgent: Your account has been compromised. Click here to secure it.' -> Prediction: Spam
Email: 'Meeting reminder for tomorrow at 10 AM.' -> Prediction: Ham
Email: 'Buy one get one free on all our products!' -> Prediction: Spam

Top 15 fea

## Transformer example

In [5]:
# use transformer for Enron spam task..
# e2e example
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

# 1. Load the data
df = pd.read_csv("./enron_spam_data.csv")

# do some sampling, good to see what happens as you sample more and more
df = df.sample(frac=0.1)

df['text'] = df['Subject'].fillna(' ') + df['Message'].fillna(' ')

# dirty trick to make the labels int 0/1
df['labels'] = (df['Spam/Ham']=='spam')*1

df = df[['text', 'labels']]


# 2. Split the dataset
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset format (if your data is in a different format)
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

# 3. Choose an Encoder-Based Transformer Model
model_name = "bert-base-uncased"

# 4. Load the Pre-trained Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 5. Preprocess the Data
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Remove the original text column as the model uses tokenized inputs
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_eval_dataset = tokenized_eval_dataset.remove_columns(["text"])


# Set the format to PyTorch tensors
tokenized_train_dataset.set_format("torch")
tokenized_eval_dataset.set_format("torch")

# 6. Define Evaluation Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 7. Configure Training Arguments
training_args = TrainingArguments(
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none" # Remove if you want to integrate with TensorBoard or other logging
)

# 8. Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 9. Train the Model
trainer.train()

# 10. Evaluate the Model on the Evaluation Set
eval_results = trainer.evaluate()
print("\nEvaluation Results:")
print(eval_results)

# 11. Inference (Example on a new text)
def predict_spam(text):
    inputs = tokenizer(text, truncation=True, padding='max_length', max_length=128, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)
    return "Spam" if predictions.item() == 1 else "Ham"

new_email = "Free iPhone! Click here to claim your prize!"
prediction = predict_spam(new_email)
print(f"\nPrediction for: '{new_email}' -> {prediction}")

new_email_ham = "Hi John, just wanted to follow up on our meeting."
prediction_ham = predict_spam(new_email_ham)
print(f"Prediction for: '{new_email_ham}' -> {prediction_ham}")



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2697 [00:00<?, ? examples/s]

Map:   0%|          | 0/675 [00:00<?, ? examples/s]

/Users/amarks-b/INCD/ML_course/day_2_spam/venv_day2/lib/python3.10/site-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/var/folders/vt/4l69jtfd3m5blk3vs5plrg_40000gp/T/ipykernel_49459/3628487564.py:85: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.139019,0.945185,0.942278,0.990164,0.898810
2,No log,0.064490,0.980741,0.980741,0.976401,0.985119
3,No log,0.082644,0.977778,0.977444,0.987842,0.967262



Evaluation Results:
{'eval_loss': 0.06449044495820999, 'eval_accuracy': 0.9807407407407407, 'eval_f1': 0.9807407407407407, 'eval_precision': 0.976401179941003, 'eval_recall': 0.9851190476190477, 'eval_runtime': 16.9293, 'eval_samples_per_second': 39.872, 'eval_steps_per_second': 0.65, 'epoch': 3.0}


RuntimeError: Placeholder storage has not been allocated on MPS device!